In [2]:
import os
import cv2
import numpy as np

# ============================================================
# CONFIGURACIÓN
# ============================================================
INPUT_TRAIN = "recursos/train"
INPUT_VAL = "recursos/val"

# Cambia este nombre según la versión que quieras generar
OUTPUT_BASE = "dataset_transformado_rgb"

OUTPUT_TRAIN = os.path.join(OUTPUT_BASE, "train")
OUTPUT_VAL = os.path.join(OUTPUT_BASE, "val")

IMG_SIZE = (224, 224)

# Opciones:
# "rgb"  -> mantiene color
# "gray" -> convierte a escala de grises
MODO_COLOR = "rgb"

# True  -> guarda imagen con valores 0-255 (normal, ideal para guardar en jpg/png)
# False -> no aplica normalización final de intensidad
NORMALIZAR_INTENSIDAD = True

# True -> aplica sharpen
USAR_SHARPEN = True

# True -> aplica suavizado ligero
USAR_BLUR = True


# ============================================================
# FUNCIÓN DE MEJORA VISUAL
# ============================================================
def mejorar_imagen(img, modo_color="rgb", normalizar_intensidad=True,
                   usar_blur=True, usar_sharpen=True):
    """
    Mejora visualmente una imagen:
    1) mejora contraste con CLAHE
    2) normaliza intensidad
    3) suaviza ruido ligero
    4) resalta bordes
    5) convierte a RGB o gris
    """

    # 1. Mejorar contraste con CLAHE sobre canal L en LAB
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_mejorado = clahe.apply(l)

    lab_mejorada = cv2.merge((l_mejorado, a, b))
    img = cv2.cvtColor(lab_mejorada, cv2.COLOR_LAB2BGR)

    # 2. Normalización de intensidad
    if normalizar_intensidad:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

    # 3. Suavizado ligero
    if usar_blur:
        img = cv2.GaussianBlur(img, (3, 3), 0)

    # 4. Sharpen
    if usar_sharpen:
        kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])
        img = cv2.filter2D(img, -1, kernel)

    # 5. Conversión de color final
    if modo_color == "gray":
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return img


# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================
def transformar_dataset(input_dir, output_dir, size=(224, 224),
                        modo_color="rgb",
                        normalizar_intensidad=True,
                        usar_blur=True,
                        usar_sharpen=True):
    """
    Recorre un dataset organizado por carpetas de clases,
    transforma cada imagen y la guarda manteniendo la misma estructura.
    """
    total = 0
    guardadas = 0
    errores = 0

    print(f"\nProcesando dataset: {input_dir}")

    for root, _, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")):
                continue

            total += 1
            in_path = os.path.join(root, file)

            img = cv2.imread(in_path)
            if img is None:
                errores += 1
                print(f"No se pudo leer: {in_path}")
                continue

            # Resize primero
            img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

            # Mejora visual
            img = mejorar_imagen(
                img,
                modo_color=modo_color,
                normalizar_intensidad=normalizar_intensidad,
                usar_blur=usar_blur,
                usar_sharpen=usar_sharpen
            )

            # Mantener estructura de carpetas
            rel_dir = os.path.relpath(root, input_dir)
            save_dir = os.path.join(output_dir, rel_dir)
            os.makedirs(save_dir, exist_ok=True)

            out_path = os.path.join(save_dir, file)

            # Guardar imagen
            ok = cv2.imwrite(out_path, img)
            if ok:
                guardadas += 1
            else:
                errores += 1
                print(f"No se pudo guardar: {out_path}")

    print(f"Total encontradas : {total}")
    print(f"Total guardadas   : {guardadas}")
    print(f"Errores           : {errores}")
    print(f"Salida            : {output_dir}")


# ============================================================
# EJECUCIÓN
# ============================================================

transformar_dataset(
    INPUT_TRAIN,
    OUTPUT_TRAIN,
    size=IMG_SIZE,
    modo_color=MODO_COLOR,
    normalizar_intensidad=NORMALIZAR_INTENSIDAD,
    usar_blur=USAR_BLUR,
    usar_sharpen=USAR_SHARPEN
)

transformar_dataset(
    INPUT_VAL,
    OUTPUT_VAL,
    size=IMG_SIZE,
    modo_color=MODO_COLOR,
    normalizar_intensidad=NORMALIZAR_INTENSIDAD,
    usar_blur=USAR_BLUR,
    usar_sharpen=USAR_SHARPEN
)


Procesando dataset: recursos/train
Total encontradas : 10000
Total guardadas   : 10000
Errores           : 0
Salida            : dataset_transformado_rgb\train

Procesando dataset: recursos/val
Total encontradas : 984
Total guardadas   : 984
Errores           : 0
Salida            : dataset_transformado_rgb\val
